In [191]:
from dotenv import load_dotenv
load_dotenv()

True

In [192]:
load_dotenv(override=True)

True

In [193]:
from google import genai


In [194]:
client = genai.Client()

In [195]:
import requests

In [208]:
docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()
question='Can yo list down atleast 2 course names'

In [197]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""
    print(course_url)

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)



https://datatalks.club/faq/json/stock-markets-analytics-zoomcamp.json
https://datatalks.club/faq/json/mlops-zoomcamp.json
https://datatalks.club/faq/json/llm-zoomcamp.json
https://datatalks.club/faq/json/data-engineering-zoomcamp.json
https://datatalks.club/faq/json/machine-learning-zoomcamp.json
https://datatalks.club/faq/json/ai-dev-tools-zoomcamp.json


1427

In [198]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [199]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [ ]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [201]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [202]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [203]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()



In [219]:
def llm(instructions, user_prompt, model):
    message_history = [
        {"type": "text", "text": user_prompt}
    ]
    
    # message_history = f"{instructions}\n\n{user_prompt}"
    # message_history = [
    #         {"role": "user", "parts": [{"text": user_prompt}]}
    #     ]
    # print(f"Message history: {message_history}")

    response = client.interactions.create(
        model=model,
        input=message_history
    )

    return response.output_text

In [205]:
def rag(query, model="gemini-3.5-flash-lite"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [223]:
answer = rag('can i join the course?')
print(answer)

Yes, you can still join the course. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.
